# Intermediate Real-World Projects

This notebook walks through a compact customer-churn project using generated data. It demonstrates a repeatable project structure, leakage-aware evaluation, preprocessing pipelines, and model interpretation. The data are synthetic; results are for learning only.

## 1. Imports and reproducibility


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
print('Imports ready; random seed =', RANDOM_STATE)

## 2. Generate a synthetic customer dataset

In a real project, document the approved source, extraction date, and target definition. Here we generate data so the notebook runs without downloads.

In [ ]:
n = 1200
tenure = rng.integers(1, 73, size=n)
monthly_fee = np.clip(rng.normal(70, 18, size=n), 20, 150).round(2)
support_tickets = rng.poisson(1.2, size=n)
contract_type = rng.choice(['monthly', 'annual', 'two_year'], size=n, p=[0.55, 0.30, 0.15])
payment_method = rng.choice(['card', 'bank_transfer', 'digital_wallet'], size=n, p=[0.5, 0.3, 0.2])
logit = (1.0 - 0.035 * tenure + 0.025 * (monthly_fee - 70)
         + 0.28 * support_tickets
         + np.where(contract_type == 'monthly', 0.75, 0)
         + np.where(payment_method == 'digital_wallet', 0.12, 0))
probability = 1 / (1 + np.exp(-logit))
churn = rng.binomial(1, probability)
df = pd.DataFrame({
    'tenure_months': tenure,
    'monthly_fee': monthly_fee,
    'support_tickets': support_tickets,
    'contract_type': contract_type,
    'payment_method': payment_method,
    'churn': churn,
})
# Add a small amount of missingness to practice a pipeline imputer.
df.loc[rng.choice(df.index, size=35, replace=False), 'monthly_fee'] = np.nan
df.head()

## 3. Audit the data


In [ ]:
print('Shape:', df.shape)
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().to_frame('missing_count'))
print('Duplicate rows:', df.duplicated().sum())
display(df['churn'].value_counts().rename('count').to_frame())
print('Churn prevalence:', round(df['churn'].mean(), 3))

## 4. Explore target and selected features

These charts describe this generated sample; do not treat them as evidence about real customers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df['tenure_months'].hist(ax=axes[0], bins=15)
axes[0].set_title('Tenure distribution')
axes[0].set_xlabel('Months')
df.groupby('contract_type')['churn'].mean().sort_values().plot(kind='bar', ax=axes[1])
axes[1].set_title('Observed churn rate by contract')
axes[1].set_ylabel('Churn rate')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 5. Define features and split the data

All predictors below are assumed to be available at prediction time. In real data, explicitly verify timing and exclude post-outcome fields.

In [ ]:
target = 'churn'
X = df.drop(columns=[target])
y = df[target]
numeric_features = ['tenure_months', 'monthly_fee', 'support_tickets']
categorical_features = ['contract_type', 'payment_method']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Train prevalence:', round(y_train.mean(), 3), 'Test prevalence:', round(y_test.mean(), 3))

## 6. Establish a simple baseline


In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print('Baseline accuracy:', round(accuracy_score(y_test, baseline_pred), 3))
print('Baseline churn recall:', round(recall_score(y_test, baseline_pred, zero_division=0), 3))

## 7. Build a leakage-aware preprocessing and modeling pipeline

The imputer, scaler, and encoder are fit on training data as part of the pipeline. `handle_unknown='ignore'` makes the encoder robust to categories not seen during training.

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])
model = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
metrics = {
    'accuracy': accuracy_score(y_test, pred),
    'precision': precision_score(y_test, pred, zero_division=0),
    'recall': recall_score(y_test, pred, zero_division=0),
    'f1': f1_score(y_test, pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, proba),
}
pd.Series(metrics, name='score').round(3).to_frame()

## 8. Inspect errors and the confusion matrix


In [ ]:
cm = confusion_matrix(y_test, pred, labels=[0, 1])
print('Rows = actual [0, 1], columns = predicted [0, 1]')
print(cm)
print(classification_report(y_test, pred, zero_division=0))

## 9. Explore a threshold decision

The default classifier threshold is not automatically optimal. Choose a threshold on validation data based on the cost of false positives, false negatives, and intervention capacity. This cell demonstrates metric calculation only; do not tune the final test set repeatedly.

In [ ]:
threshold = 0.60
threshold_pred = (proba >= threshold).astype(int)
print('Threshold:', threshold)
print('Precision:', round(precision_score(y_test, threshold_pred, zero_division=0), 3))
print('Recall:', round(recall_score(y_test, threshold_pred, zero_division=0), 3))
print('Flagged for review:', int(threshold_pred.sum()), 'of', len(threshold_pred))

## 10. Create a compact project summary


In [ ]:
summary = {
    'rows': len(df),
    'features': X.shape[1],
    'target_prevalence': round(float(y.mean()), 3),
    'baseline_accuracy': round(float(accuracy_score(y_test, baseline_pred)), 3),
    'model_accuracy': round(float(metrics['accuracy']), 3),
    'model_precision': round(float(metrics['precision']), 3),
    'model_recall': round(float(metrics['recall']), 3),
    'model_roc_auc': round(float(metrics['roc_auc']), 3),
}
pd.Series(summary, name='value').to_frame()

## 11. Turn results into a business recommendation

Use this template and fill it with the results from your run:

- **Decision:** what action will the team take?
- **Evidence:** which holdout metrics and error patterns matter?
- **Capacity:** how many cases can be reviewed or contacted?
- **Costs:** what is the cost of a false positive and a missed churner?
- **Limitations:** synthetic data, assumptions, selection effects, and drift.
- **Next step:** run a controlled pilot and measure incremental retention before claiming business impact.

## Extension ideas

1. Add cross-validation and compare a second model.
2. Build a house-price regression pipeline and report MAE.
3. Apply scaled K-means to synthetic customer behavior and profile segments.
4. Add a time-based sales dataset and compare forecast baselines.
5. Save a model artifact and document versioning and monitoring requirements.